Example code for regridding CMIP6 data as used in CMIP6 WUS winter warming paper.

In [47]:
import netCDF4
#import h5netcdf
import xarray as xr
import scipy
import os
import xesmf as xe
import numpy as np
import xeofs as xeo
import scipy.stats as stats
import warnings
import json

In [ ]:
modmembers_prw_ssp245 = {}

# open dictionary of models and ensemble members as in {model_name1:[member_id1, member_id2,...], model_name2:[...], ...}}
with open('/home/tessj/wus_temp_trends/modmembers_prw_ssp245.json', 'r') as file:
        modmembers_prw_ssp245 = json.load(file)


In [ ]:
    # variable name
    var = 'prw'
    regrid_periodic = False
    scenario = 'ssp245' #'historical'
    dates = {'ssp245' : '20150101-20241231', 'historical':'18500101-20141231'}[scenario]
    modmember_dict = modmembers_prw_ssp245
    
    # check if regridded data is already on disk somewhere
    for model in list(modmember_dict.keys())[:]:
        if model not in os.listdir('/d5/tessj/data/CMIP6/for_tess/'):
            os.mkdir(f'/d5/tessj/data/CMIP6/for_tess/{model}')
    #try: 
        already_exists = os.listdir(f'/d3/tessj/data/CMIP6/for_tess/{model}')
        already_exists_d1 = os.listdir(f'/d1/tessj/data/CMIP6/for_tess/{model}')
        already_exists_d4 = os.listdir(f'/d4/tessj/data/CMIP6/for_tess/{model}')
        already_exists_d5 = os.listdir(f'/d5/tessj/data/CMIP6/for_tess/{model}')


        skip = 0
        for member_id in modmember_dict[model]:
            if regrid_periodic==False:
                regridded_path = f'{var}_{table_id}_{model}_{scenario}_{member_id}_{dates}_regrid.nc'
            elif regrid_periodic==True:
                regridded_path = f'{var}_{table_id}_{model}_{scenario}_{member_id}_{dates}_regridp.nc'
            
            if regridded_path in (already_exists + already_exists_d1 + already_exists_d4 +already_exists_d5): 
                print(model + ' ' + member_id + 'model already regridded and saved, skip')
                skip = 1
                
            elif f'{var}_{table_id}_{model}_{scenario}_{member_id}_{dates}.nc' in (already_exists + already_exists_d1+ already_exists_d4+already_exists_d5): 
                if var in ['zg300', 'clt', 'w700', 'hfds', 'hurs', 'tas']:
                    testds = xr.open_dataset(f'/d1/tessj/data/CMIP6/for_tess/{model}/{var}_{table_id}_{model}_{scenario}_{member_id}_{dates}.nc')
                else:
                    testds = xr.open_dataset(f'/d5/tessj/data/CMIP6/for_tess/{model}/{var}_{table_id}_{model}_{scenario}_{member_id}_{dates}.nc')

                print('loaded '+ model + ' ' + member_id)
                
            else:
                skip = 1
                print('.nc file for ' + model + ' ' + member_id + 'does not exist, skip')
                
            if skip == 0:
                # Regrid to 1-deg grid
                # Ignore degenerate to avoid xesmf error when grids have 
                # near singularities (as is sometimes the case in ocean grids
                # with weird poles. Error code 506 or something.)
                rgrd = xe.Regridder(testds,sample_grid,'bilinear',ignore_degenerate=True, periodic=regrid_periodic)
                
                testds_rg = rgrd(testds)
                
                print('regridded')
                drop_vars = [var for var in list(testds_rg.data_vars) if var not in [var]]
                
                if model not in os.listdir('/d5/tessj/data/CMIP6/for_tess/'):
                    os.mkdir(f'/d5/tessj/data/CMIP6/for_tess/{model}')
                testds_rg.drop(drop_vars).to_netcdf(f'/d5/tessj/data/CMIP6/for_tess/{model}/{var}_{table_id}_{model}_{scenario}_{member_id}_{dates}_regrid.nc')
                print('saved out d5')
            skip = 0
                             
